# 01 · Data Understanding
**Netflix Content Analytics — Phase 1**

> *"Before you clean it, understand it. Before you analyse it, clean it."*

This notebook is a **read-only investigation**. We do not change a single value here.
Our job is to act like a detective: find out what the data contains, what it means,
and what is broken — and write all of it down.

**What you will learn**
1. How to load a dataset reproducibly
2. How to profile shape, types and sample records
3. How to hunt for missing values, duplicates and inconsistent formats
4. How to turn those findings into a **data dictionary** and an **issue log** that drives Phase 2

⚠️ **Framing to keep in mind all project long:** this dataset is *catalog metadata* —
which titles exist and when they were added. It contains **no viewership, revenue,
or subscriber data**, so it can describe *what Netflix offers*, never *what people watch*.

## Step 1 · Imports and display settings
- `pandas` is our table engine; `numpy` helps with numeric work.
- `pathlib.Path` builds file paths that work on Windows, macOS and Linux — reproducibility matters.
- Pandas hides wide columns by default. `cast` and `description` are long, so we widen the display.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 3.0.3 | numpy 2.5.1


## Step 2 · Load the raw CSV
The notebook lives in `notebooks/`, so the data is one folder up (`..`).
We name the DataFrame `netflix_raw` so it is always obvious which version we are looking at.

In [2]:
RAW_DATA_PATH = Path("..") / "data" / "raw" / "netflix_titles.csv"

netflix_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded: {RAW_DATA_PATH}")

Loaded: ..\data\raw\netflix_titles.csv


## Step 3 · First look — shape, columns, data types
- **Shape** is our baseline. Every row we remove later must be explained against this number.
- **Data types** reveal hidden problems: if a date is stored as text, we must convert it.

💡 In pandas 3.x text columns show as `str` (older tutorials show `object` — same idea).

In [3]:
n_rows, n_cols = netflix_raw.shape
print(f"Rows: {n_rows:,} | Columns: {n_cols}\n")
print(netflix_raw.dtypes)

Rows: 8,807 | Columns: 12

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object


**🔍 What we see:** only `release_year` is numeric. `date_added` and `duration` are stored as
text even though one is a date and the other contains a number — both will need work in Phase 2.

## Step 4 · Look at real records
`head()` shows the top of the file, which may be sorted and unrepresentative.
`sample()` gives a fairer view; `random_state=42` makes it reproducible.

In [4]:
netflix_raw.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirst..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, ...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets o..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari,...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adve...","To protect his family from a powerful drug lord, skilled..."
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the inc..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaa...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies",In a city of coaching centers known to train India’s fin...


In [5]:
netflix_raw.sample(5, random_state=42)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
4970,s4971,Movie,"Game Over, Man!",Kyle Newacheck,"Adam DeVine, Anders Holm, Blake Anderson, Utkarsh Ambudk...",United States,"March 23, 2018",2018,TV-MA,102 min,"Action & Adventure, Comedies",Three buddies with big dreams go from underachieving sla...
3362,s3363,Movie,Arsenio Hall: Smart & Classy,Brian Volk-Weiss,Arsenio Hall,United States,"October 29, 2019",2019,TV-MA,63 min,Stand-Up Comedy,"In his first stand-up special, Arsenio Hall discusses ge..."
5494,s5495,TV Show,Kazoops!,NaN,"Reece Pockney, Scott Langley, Alex Babic, Gemma Harvey, ...",Australia,"May 5, 2017",2017,TV-Y,3 Seasons,Kids' TV,Music meets imagination in this inventive animated serie...
1688,s1689,TV Show,We Are the Champions,NaN,NaN,United States,"November 17, 2020",2020,TV-MA,1 Season,"Docuseries, Reality TV","Explore an array of unique competitions, from the quirky..."
1349,s1350,TV Show,"Pablo Escobar, el patrón del mal",NaN,"Andrés Parra, Angie Cepeda, Cecilia Navia, Vicky Hernánd...",Colombia,"February 3, 2021",2012,TV-MA,1 Season,"Crime TV Shows, International TV Shows, Spanish-Language...",From his days as a petty thief to becoming head of a dru...


**🧠 Analyst thinking — patterns to notice:**
- `country`, `cast`, `director` and `listed_in` can hold **several values in one cell**, separated by commas.
- `date_added` looks like `"September 25, 2021"` — human-readable text, not a date.
- `duration` means **minutes for movies** but **seasons for TV shows** — two different units in one column!

## Step 5 · Missing values
We count missing values per column **and** express them as a percentage —
10 missing out of 8,807 is trivial; 2,600 missing is a structural issue.

In [6]:
missing_summary = (
    pd.DataFrame({
        "missing_count": netflix_raw.isna().sum(),
        "missing_pct": (netflix_raw.isna().mean() * 100).round(2),
    })
    .sort_values("missing_count", ascending=False)
)
missing_summary

,missing_count,missing_pct
director,2634,29.91
country,831,9.44
cast,825,9.37
date_added,10,0.11
rating,4,0.05
duration,3,0.03
show_id,0,0.00
type,0,0.00
title,0,0.00
release_year,0,0.00


A missing value only becomes meaningful when we ask **"missing for whom?"**
Let's split the missing percentage by content type.

In [7]:
missing_by_type = (
    netflix_raw.drop(columns="type")
    .isna()
    .groupby(netflix_raw["type"])
    .mean()
    .mul(100)
    .round(1)
    .T
)
missing_by_type[missing_by_type.sum(axis=1) > 0]

type,Movie,TV Show
director,3.1,91.4
cast,7.7,13.1
country,7.2,14.6
date_added,0.0,0.4
rating,0.0,0.1


**🔍 What this tells us:**
- `director` is missing for the vast majority of **TV shows** but only a small share of movies.
  That is structural, not random: series often have many episode directors, so the field is left blank.
  ➜ We should **not** "fill" directors — we'll label them as not listed.
- All missing `date_added` values are TV shows (a small number) — too few to matter, but we must not invent dates.
- `country` is missing for a meaningful minority of titles, so country charts must disclose an "Unknown" share.

## Step 6 · Duplicates — investigate before deleting
Duplicates come in layers. We check from strictest to loosest:
1. Fully identical rows
2. Repeated `show_id` (the primary key)
3. Same title + type + release year
4. Same title after **normalising** case and invisible whitespace

In [8]:
print("1. Fully identical rows:        ", netflix_raw.duplicated().sum())
print("2. Repeated show_id:            ", netflix_raw["show_id"].duplicated().sum())
print("3. Same title + type + year:    ", netflix_raw.duplicated(["title", "type", "release_year"]).sum())

# Normalise: replace non-breaking spaces, trim, lowercase
normalised_title = (
    netflix_raw["title"]
    .str.replace(" ", " ", regex=False)
    .str.strip()
    .str.lower()
)
print("4. Same normalised title:       ", normalised_title.duplicated().sum())

1. Fully identical rows:         0
2. Repeated show_id:             0
3. Same title + type + year:     0
4. Same normalised title:        6


Exact checks find nothing — but the normalised check finds some. Let's look at them side by side.

In [9]:
possible_duplicates = (
    netflix_raw[normalised_title.duplicated(keep=False)]
    .assign(normalised_title=normalised_title)
    .sort_values(["normalised_title", "show_id"])
    [["show_id", "title", "type", "release_year", "date_added", "rating", "duration", "country"]]
)
possible_duplicates

,show_id,title,type,release_year,date_added,rating,duration,country
3371,s3372,Consequences,Movie,2014,"October 25, 2019",TV-MA,106 min,Turkey
6529,s6530,Consequences,Movie,2014,"October 25, 2019",TV-MA,106 min,Turkey
5318,s5319,Death Note,Movie,2017,"August 25, 2017",TV-MA,100 min,United States
5751,s5752,DEATH NOTE,TV Show,2006,"October 15, 2016",TV-14,1 Season,Japan
303,s304,Esperando la carroza,Movie,1985,"August 5, 2021",TV-MA,95 min,Argentina
6705,s6706,Esperando La Carroza,Movie,1985,"July 15, 2018",NR,95 min,Argentina
5033,s5034,FullMetal Alchemist,Movie,2017,"February 19, 2018",TV-PG,134 min,Japan
5095,s5096,Fullmetal Alchemist,TV Show,2003,"January 1, 2018",TV-14,1 Season,Japan
159,s160,Love in a Puff,Movie,2010,"September 1, 2021",TV-MA,103 min,Hong Kong
7345,s7346,Love In A Puff,Movie,2010,"August 1, 2018",TV-MA,103 min,Hong Kong


**🧠 Reading the evidence pair by pair:**

| Pair | Verdict | Why |
|---|---|---|
| *Consequences* (s3372 / s6530) | **True duplicate** | Identical in every field — one title just has an invisible non-breaking space at the end |
| *Death Note*, *Fullmetal Alchemist* | **Not duplicates** | One is a Movie, the other a TV Show, different years |
| *Esperando la carroza*, *Love in a Puff*, *Sin senos sí hay paraíso* | **Ambiguous** | Same title/year/country, but different `date_added` (and one has a different rating) — possibly re-listed titles or separate catalogue entries |

➜ **Decision for Phase 2:** remove only the proven duplicate. Keep the ambiguous ones (they have
different `show_id`s and add dates — deleting them would be a guess) and **document** them as a limitation.

In [10]:
# Proof that the "Consequences" pair is identical except for the hidden character
pair = netflix_raw[netflix_raw["show_id"].isin(["s3372", "s6530"])]
print([repr(t) for t in pair["title"]])
print("Columns that differ:", [c for c in pair.columns if pair[c].nunique(dropna=False) > 1])

["'Consequences'", "'Consequences\\xa0'"]
Columns that differ: ['show_id', 'title']


## Step 7 · Inconsistent formats & data-quality traps
Now we check each column for values that *look* fine but would break our analysis.

### 7a · `rating` contains durations?!

In [11]:
netflix_raw["rating"].value_counts(dropna=False)

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
NaN            4
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64

In [12]:
netflix_raw[netflix_raw["rating"].str.contains("min", na=False)][["show_id", "title", "type", "rating", "duration"]]

,show_id,title,type,rating,duration
5541,s5542,Louis C.K. 2017,Movie,74 min,NaN
5794,s5795,Louis C.K.: Hilarious,Movie,84 min,NaN
5813,s5814,Louis C.K.: Live at the Comedy Store,Movie,66 min,NaN


**🔍 A classic shifted-column error.** For these rows the movie length landed in `rating` and `duration`
is empty. These are also exactly the rows with missing `duration`.
➜ Phase 2: move the value to `duration` and mark the rating as unknown.

### 7b · `date_added` — hidden whitespace and format

In [13]:
date_text = netflix_raw["date_added"].dropna()
has_extra_spaces = date_text != date_text.str.strip()
print("Dates with leading/trailing spaces:", has_extra_spaces.sum())
print("Example:", repr(date_text[has_extra_spaces].iloc[0]))

parsed = pd.to_datetime(date_text.str.strip(), format="%B %d, %Y", errors="coerce")
print("Dates that fail to parse after stripping:", parsed.isna().sum())
print("Date range:", parsed.min().date(), "→", parsed.max().date())

Dates with leading/trailing spaces: 88
Example: ' August 4, 2017'
Dates that fail to parse after stripping: 0
Date range: 2008-01-01 → 2021-09-25


**🔍** A few dozen dates start with a space (invisible in Excel!). After stripping, **every** date parses
with one format. Notice the date range ends in **September 2021** → 2021 is a **partial year**. We must
never compare 2021 totals with full years without saying so.

### 7c · `duration` — two units in one column

In [14]:
duration_unit = netflix_raw["duration"].str.split(" ").str[1]
pd.crosstab(netflix_raw["type"], duration_unit, dropna=False)

duration,Season,Seasons,min,NaN
type,,,,
Movie,0,0,6128,3
TV Show,1793,883,0,0


**🔍** Movies → `min`, TV shows → `Season`/`Seasons` (singular vs plural). The units line up perfectly
with `type` — reassuring. ➜ Phase 2: split into `duration_value` (number) + `duration_unit` (`minutes` / `seasons`).

### 7d · Multi-value columns (`country`, `listed_in`)

In [15]:
for column in ["country", "listed_in", "director", "cast"]:
    share_multi = netflix_raw[column].str.contains(",", na=False).mean() * 100
    print(f"{column:<10} rows with >1 value: {share_multi:5.1f}%")

country    rows with >1 value:  15.0%
listed_in  rows with >1 value:  77.1%
director   rows with >1 value:   7.0%
cast       rows with >1 value:  80.6%


In [16]:
# Stray commas produce empty country names after splitting
netflix_raw[netflix_raw["country"].str.match(r"^\s*,|,\s*$", na=False)][["show_id", "title", "country"]]

,show_id,title,country
193,s194,D.P.,", South Korea"
365,s366,Eyes of a Thief,", France, Algeria"
1192,s1193,The Present,"United Kingdom,"
2224,s2225,Funan,"France, Belgium, Luxembourg, Cambodia,"
4653,s4654,City of Joy,"United States,"
5925,s5926,Virunga,"United Kingdom,"
7007,s7008,Hotel Pacific,"Poland,"


**🔍** Most titles list several genres, and a meaningful share list several countries. If we count the raw
text `"United States, India"` as one country, our country rankings would be wrong.
➜ Phase 2: build **normalised (long) tables** — one row per title-country and per title-genre — and drop
empty strings left by stray commas.

### 7e · Hidden characters & numeric sanity checks

In [17]:
hidden_char_pattern = "[ ​‌‍﻿]"
for column in netflix_raw.select_dtypes("str").columns:
    count = netflix_raw[column].str.contains(hidden_char_pattern, na=False).sum()
    if count:
        print(f"{column:<12} values containing non-breaking / zero-width spaces: {count}")

title        values containing non-breaking / zero-width spaces: 10
cast         values containing non-breaking / zero-width spaces: 1
description  values containing non-breaking / zero-width spaces: 9


In [18]:
netflix_raw["release_year"].describe().round(1)

count    8807.0
mean     2014.2
std         8.8
min      1925.0
25%      2013.0
50%      2017.0
75%      2019.0
max      2021.0
Name: release_year, dtype: float64

`release_year` ranges from the 1920s to 2021 — plausible (classic films exist). No impossible values.

In [19]:
netflix_raw["listed_in"].str.split(",").explode().str.strip().nunique(), \
netflix_raw["country"].str.split(",").explode().str.strip().replace("", np.nan).dropna().nunique()

(42, 122)

There are a few dozen distinct listed categories and over a hundred distinct countries once split.
Note that categories are **type-specific** ("Dramas" vs "TV Dramas", "International Movies" vs
"International TV Shows") and some are not genres at all (e.g. "International Movies", "British TV Shows").

## Step 8 · Data dictionary
A data dictionary explains every column so anyone (including future-you) can use the data correctly.

In [20]:
data_dictionary = pd.DataFrame([
    ("show_id",      "str", "Unique identifier for each catalogue entry (e.g. s1)", "Primary key; no duplicates"),
    ("type",         "str", "Content type: Movie or TV Show", "Complete"),
    ("title",        "str", "Title of the movie or show", "A few titles contain hidden non-breaking spaces"),
    ("director",     "str", "Director(s), comma-separated", "Mostly missing for TV shows (structural)"),
    ("cast",         "str", "Main cast members, comma-separated", "Partially missing"),
    ("country",      "str", "Country/countries of production, comma-separated", "Partially missing; stray commas"),
    ("date_added",   "str", "Date the title was added to Netflix (text, e.g. 'September 25, 2021')", "Text not date; leading spaces; few missing; ends Sep-2021"),
    ("release_year", "int", "Original release year of the title", "Complete; plausible range"),
    ("rating",       "str", "Content maturity rating (TV-MA, PG-13, ...)", "3 rows contain durations; few missing"),
    ("duration",     "str", "Minutes for movies ('90 min'); seasons for TV shows ('2 Seasons')", "Mixed units; 3 missing (shifted into rating)"),
    ("listed_in",    "str", "Netflix categories/genres, comma-separated", "Multi-valued; type-specific labels"),
    ("description",  "str", "Short plot summary", "Complete; not used for quantitative analysis"),
], columns=["column", "raw_dtype", "meaning", "quality_notes"])

data_dictionary["missing_count"] = data_dictionary["column"].map(netflix_raw.isna().sum())
data_dictionary

,column,raw_dtype,meaning,quality_notes,missing_count
0,show_id,str,Unique identifier for each catalogue entry (e.g. s1),Primary key; no duplicates,0
1,type,str,Content type: Movie or TV Show,Complete,0
2,title,str,Title of the movie or show,A few titles contain hidden non-breaking spaces,0
3,director,str,"Director(s), comma-separated",Mostly missing for TV shows (structural),2634
4,cast,str,"Main cast members, comma-separated",Partially missing,825
5,country,str,"Country/countries of production, comma-separated",Partially missing; stray commas,831
6,date_added,str,"Date the title was added to Netflix (text, e.g. 'Septemb...",Text not date; leading spaces; few missing; ends Sep-2021,10
7,release_year,int,Original release year of the title,Complete; plausible range,0
8,rating,str,"Content maturity rating (TV-MA, PG-13, ...)",3 rows contain durations; few missing,4
9,duration,str,Minutes for movies ('90 min'); seasons for TV shows ('2 ...,Mixed units; 3 missing (shifted into rating),3


In [21]:
# Save the dictionary so it can be linked from the README
REPORTS_DIR = Path("..") / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

# Build a Markdown table by hand (avoids an extra dependency)
header = "| " + " | ".join(data_dictionary.columns) + " |"
divider = "|" + "---|" * len(data_dictionary.columns)
rows = ["| " + " | ".join(str(v) for v in row) + " |" for row in data_dictionary.itertuples(index=False)]
(REPORTS_DIR / "data_dictionary.md").write_text(
    "# Data Dictionary — raw `netflix_titles.csv`\n\n" + "\n".join([header, divider, *rows]) + "\n",
    encoding="utf-8",
)
print("Saved reports/data_dictionary.md")

Saved reports/data_dictionary.md


## Step 9 · Issue log → the plan for Phase 2
Every issue we found becomes a documented cleaning decision. This table is what separates a
professional analysis from "I ran dropna() and hoped for the best".

| # | Issue | Evidence | Planned action (Phase 2) |
|---|---|---|---|
| 1 | `date_added` stored as text, some with leading spaces | Step 7b | Strip, parse to date with explicit format |
| 2 | `date_added` missing for a few TV shows | Step 5 | Keep as missing (don't invent dates) |
| 3 | Duration values shifted into `rating` | Step 7a | Move to `duration`; set rating to `Unknown` |
| 4 | `rating` missing for a few rows | Step 5 | Label `Unknown` |
| 5 | `duration` mixes minutes and seasons | Step 7c | Split into `duration_value` + `duration_unit` |
| 6 | `director` / `cast` missing (structural for TV) | Step 5 | Label `Not Listed`; don't impute |
| 7 | `country` missing | Step 5 | Label `Unknown`; disclose in charts |
| 8 | Multi-valued `country` / `listed_in`, stray commas | Step 7d | Bridge tables, one row per value |
| 9 | Hidden non-breaking spaces in text | Step 7e | Replace with normal spaces and trim |
| 10 | One true duplicate (*Consequences*) | Step 6 | Drop the duplicate entry |
| 11 | Three ambiguous same-title pairs | Step 6 | Keep; document as limitation |
| 12 | 2021 is a partial year (data ends Sep-2021) | Step 7b | Flag in every time-based chart |

✅ **Phase 1 complete.** Next: `02_data_cleaning.ipynb`.